# 04 — Elliptic Bitcoin Dataset Deneyi

**Amaç:** Modelin farklı bir domain'de (Bitcoin fraud) de çalıştığını göstermek.

Elliptic zaten homojen graph formatında geliyor, bu yüzden:
- GCN baseline direkt çalışır
- Hybrid model için tek node type'lı hetero graph'a çeviririz

In [ ]:
# Colab setup — run once
import os, sys

if 'google.colab' in str(get_ipython()):
    # Mount Drive for data
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone repo if not present
    if not os.path.exists('/content/graduation-p'):
        !git clone https://github.com/berk0vic/graduation-p.git /content/graduation-p
    os.chdir('/content/graduation-p/notebooks')
    sys.path.insert(0, '/content/graduation-p')

    # Link data from Drive
    import subprocess
    os.makedirs('../data/raw/elliptic', exist_ok=True)
    subprocess.run(['ln', '-sfn',
        '/content/drive/MyDrive/graduation_data/elliptic_bitcoin_dataset',
        '../data/raw/elliptic/elliptic_bitcoin_dataset'], check=True)
    print('Drive linked.')
else:
    print('Running locally.')


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## 1. Veri Yükleme

In [ ]:
from src.data.elliptic_loader import build_pyg_graph

graph = build_pyg_graph()

print(f'Nodes: {graph.x.shape[0]:,}')
print(f'Features: {graph.x.shape[1]}')
print(f'Edges: {graph.edge_index.shape[1]:,}')
print()

# Etiket dağılımı
labeled_mask = graph.y >= 0  # -1 = unknown, bunları eğitimde kullanmayacağız
y_labeled = graph.y[labeled_mask]
n_illicit = (y_labeled == 1).sum().item()
n_licit = (y_labeled == 0).sum().item()
n_unknown = (graph.y == -1).sum().item()

print(f'Labeled: {len(y_labeled):,} (illicit: {n_illicit:,}, licit: {n_licit:,})')
print(f'Unknown: {n_unknown:,}')
print(f'Fraud oranı (labeled): {100*n_illicit/len(y_labeled):.1f}%')

## 2. Train/Test Split

In [ ]:
# Labeled node'ları train / val / test'e böl (70 / 15 / 15)
from sklearn.model_selection import train_test_split

labeled_indices = labeled_mask.nonzero(as_tuple=True)[0].numpy()
y_labeled_np = y_labeled.numpy()

# First split: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    labeled_indices, test_size=0.30, stratify=y_labeled_np, random_state=42
)
y_temp = graph.y[temp_idx].numpy()

# Second split: 50/50 of temp → val and test (each 15%)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=y_temp, random_state=42
)

train_mask = torch.zeros(graph.x.shape[0], dtype=torch.bool)
val_mask   = torch.zeros(graph.x.shape[0], dtype=torch.bool)
test_mask  = torch.zeros(graph.x.shape[0], dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx]     = True
test_mask[test_idx]   = True

print(f'Train: {train_mask.sum().item():,} (fraud: {graph.y[train_mask].sum().item():,})')
print(f'Val:   {val_mask.sum().item():,}   (fraud: {graph.y[val_mask].sum().item():,})')
print(f'Test:  {test_mask.sum().item():,}  (fraud: {graph.y[test_mask].sum().item():,})')


## 3. GCN Baseline

In [ ]:
from src.models.baselines import GCNBaseline
from src.training.losses import focal_loss
from src.evaluation.metrics import compute_metrics, print_report

gcn = GCNBaseline(in_channels=graph.x.shape[1], hidden=64, out=32).to(device)
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.001, weight_decay=1e-4)

x_dev = graph.x.to(device)
edge_dev = graph.edge_index.to(device)
y_dev = graph.y.float().to(device)

best_state = None
best_loss = float('inf')

for epoch in range(1, 101):
    gcn.train()
    optimizer.zero_grad()
    logits = gcn(x_dev, edge_dev)
    loss = focal_loss(logits[train_mask.to(device)], y_dev[train_mask.to(device)])
    loss.backward()
    optimizer.step()
    
    if loss.item() < best_loss:
        best_loss = loss.item()
        best_state = {k: v.cpu().clone() for k, v in gcn.state_dict().items()}
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | loss: {loss.item():.4f}')

gcn.load_state_dict(best_state)

# Test
gcn.eval()
with torch.no_grad():
    scores = torch.sigmoid(gcn(x_dev, edge_dev)).cpu().numpy()

test_y = graph.y[test_mask].numpy()
test_scores = scores[test_mask.numpy()]

print('\n=== GCN Baseline (Elliptic) ===')
gcn_metrics = compute_metrics(test_y, test_scores)
print_report(test_y, test_scores)

## 4. Hybrid GAT+VAE (Elliptic)

Elliptic homojen graph. Hybrid model için tek node type'lı HeteroData'ya çeviriyoruz.

In [ ]:
from torch_geometric.data import HeteroData
from src.models.hybrid_model import HybridGATVAE
import yaml

# Homojen graph'ı tek-tipli hetero graph'a çevir
hetero = HeteroData()
hetero['transaction'].x = graph.x
hetero['transaction'].y = graph.y
hetero['transaction'].train_mask = train_mask
hetero['transaction'].val_mask   = val_mask
hetero['transaction'].test_mask  = test_mask
hetero['transaction', 'linked_to', 'transaction'].edge_index = graph.edge_index

print('HeteroData:')
print(f'  Node types: {hetero.node_types}')
print(f'  Edge types: {hetero.edge_types}')
print(f'  Transaction features: {hetero["transaction"].x.shape}')


In [ ]:
with open('../configs/default.yaml') as f:
    cfg = yaml.safe_load(f)

in_channels = {'transaction': graph.x.shape[1]}

model = HybridGATVAE(
    metadata=hetero.metadata(),
    in_channels=in_channels,
    raw_txn_dim=graph.x.shape[1],
    gat_hidden=cfg['model']['gat_hidden'],
    gat_out=cfg['model']['gat_out'],
    gat_heads=cfg['model']['gat_heads'],
    gat_layers=cfg['model']['gat_layers'],
    vae_latent=cfg['model']['vae_latent'],
    vae_hidden=cfg['model']['vae_hidden'],
    dropout=cfg['model']['dropout'],
)

print(f'Toplam parametre: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    lr=cfg['training']['lr'],
    weight_decay=cfg['training']['weight_decay'],
    lambda1=cfg['training']['lambda1'],
    lambda2=cfg['training']['lambda2'],
    focal_alpha=cfg['training']['focal_alpha'],
    focal_gamma=cfg['training']['focal_gamma'],
    device=str(device),
)

print(f'Eğitim başlıyor... (100 epoch)')
history = trainer.fit(hetero, val_data=hetero, epochs=100)
print('Eğitim tamamlandı!')


In [ ]:
# Hybrid model test sonuçları
eval_result = trainer.evaluate(hetero)
hybrid_scores = eval_result['fraud_scores'].numpy()

hybrid_test_scores = hybrid_scores[test_mask.numpy()]

print('=== Hybrid GAT+VAE (Elliptic) ===')
hybrid_metrics = compute_metrics(test_y, hybrid_test_scores)
print_report(test_y, hybrid_test_scores)

## 5. Karşılaştırma

In [ ]:
results = pd.DataFrame({
    'Model': ['GCN Baseline', 'Hybrid GAT+VAE'],
    'F1 (Fraud)': [gcn_metrics['f1_fraud'], hybrid_metrics['f1_fraud']],
    'Recall (Fraud)': [gcn_metrics['recall_fraud'], hybrid_metrics['recall_fraud']],
    'Precision (Fraud)': [gcn_metrics['precision_fraud'], hybrid_metrics['precision_fraud']],
    'AUC-ROC': [gcn_metrics['roc_auc'], hybrid_metrics['roc_auc']],
})

print(results.to_string(index=False))
results.to_csv('../results/tables/elliptic_results.csv', index=False)
print('\nKaydedildi: results/tables/elliptic_results.csv')

In [ ]:
# Model kaydet
torch.save(model.state_dict(), '../results/models/hybrid_gatvae_elliptic.pt')
print('Model kaydedildi.')